Install 

In [0]:
# %pip install openmeteo-requests
# %pip install requests-cache retry-requests numpy pandas

In [0]:
# %restart_python

Usage

In [0]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# --- Spark & Delta Setup ---
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

# Initialize SparkSession configured for Delta Lake
spark = SparkSession.builder \
    .appName("OpenMeteoMerge") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Target table name in the data catalog
table_name = "weather_openmeteo.bronze.weather_raw"

# --- API Client Setup ---
cache_session = requests_cache.CachedSession('/tmp/.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# --- Configuration ---
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": [42.6828, 42.5084, 43.0731],
    "longitude": [-89.0187, -89.0318, -89.4012],
    "hourly": ["temperature_2m", "weather_code"],
    "models": "gfs_seamless",
    "past_days": 1,
    "temperature_unit": "fahrenheit",
}

# Fetch weather data from the API
responses = openmeteo.weather_api(url, params=params)

# Instantiate the DeltaTable object using the catalog name (forName)
# Note: The table must already exist in the catalog for this to work
delta_table = DeltaTable.forName(spark, table_name)

# Process all locations
for response in responses:
    lat = round(response.Latitude(), 4)
    lon = round(response.Longitude(), 4)
    
    # Extract hourly data
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_weather_code = hourly.Variables(1).ValuesAsNumpy()
    
    # Create the Pandas DataFrame
    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "latitude": lat,
        "longitude": lon,
        "temperature_2m": hourly_temperature_2m,
        "weather_code": hourly_weather_code
    }
    df_Weather = pd.DataFrame(data=hourly_data)
    
    # Convert Pandas DataFrame to Spark DataFrame
    df_spark_source = spark.createDataFrame(df_Weather)

    # Execute incremental Merge for each location
    (
        delta_table.alias("target")
        .merge(
            df_spark_source.alias("source"),
            "target.date = source.date AND target.latitude = source.latitude AND target.longitude = source.longitude"
        )
        #.whenMatchedUpdate(set={
        #    "temperature_2m": "source.temperature_2m",
        #    "weather_code": "source.weather_code"
        #})
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Processing and Merge completed successfully!")


    # print(df_Weather.head())

    # spark.createDataFrame(df_Weather).write \
    # .format("delta") \
    # .mode("overwrite") \
    # .saveAsTable("weather_openmeteo.bronze.weather_raw")

Create Medalians tables if not exist

In [0]:
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN weather_openmeteo").show()

spark.sql("""
CREATE SCHEMA IF NOT EXISTS weather_openmeteo.bronze
""")

spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN weather_openmeteo").show()


Salvar o dataframe como tabela Delta